# 階段三：特徵工程

本 Notebook 負責：
1. 建立 Lag 特徵（前1、2、3個月住房率，去年同月）
2. 建立日曆特徵（季節、假日、月份編碼）
3. 建立旅客結構特徵
4. 建立旅館屬性特徵
5. 輸出完整特徵資料供模型訓練

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

from features import build_all_features, get_feature_columns

PROCESSED_DIR = Path('..') / 'data' / 'processed'

# 載入清理後資料
df = pd.read_csv(PROCESSED_DIR / 'hotel_combined.csv', parse_dates=['year_month'])
print(f'原始資料：{df.shape}')
df.head()

## 1. 執行特徵工程

In [ ]:
# 執行全部特徵工程
df = build_all_features(df)

print(f'\n特徵工程後資料：{df.shape}')
print(f'\n新增欄位：')
for col in df.columns:
    print(f'  {col}: {df[col].dtype}')

## 2. 檢查 Lag 特徵是否正確

In [ ]:
# 檢查 Lag 特徵是否正確（抽樣一間旅館）
if 'hotel_name' in df.columns:
    sample_hotel = df['hotel_name'].value_counts().index[0]
    sample = df[df['hotel_name'] == sample_hotel][['hotel_name', 'year_month', 'occupancy_rate', 'occ_lag1', 'occ_lag2', 'occ_lag3', 'occ_lag12']].head(15)
    print(f'抽樣旅館：{sample_hotel}')
    display(sample)

# 檢查缺失值比例
feature_cols = get_feature_columns()
available_cols = [c for c in feature_cols if c in df.columns]
missing_pct = df[available_cols].isnull().mean().round(4) * 100
print('\n=== 特徵缺失值比例 ===')
print(missing_pct.sort_values(ascending=False))

## 3. 儲存特徵資料

In [ ]:
# 儲存特徵資料
out_path = PROCESSED_DIR / 'hotel_features.csv'
df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'已儲存：{out_path}')
print(f'資料筆數：{len(df)}')
print(f'欄位數：{len(df.columns)}')
print(f'\n可用特徵數：{len(available_cols)}')
print(f'可用特徵：{available_cols}')

print('\n\n✅ 特徵工程完成！下一步：執行 04_modeling.ipynb 進行模型訓練')